<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/05_sre_applications/llm_finetuning/01_lora_finetuning_basics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/05_sre_applications/llm_finetuning/01_lora_finetuning_basics.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Fine-Tuning a Small LLM for Infrastructure Log Parsing

## Objectives
- Understand **LoRA (Low-Rank Adaptation)**: instead of retraining all weights we inject
  tiny trainable adapter matrices — here just **0.34 %** of the model is updated.
- Fine-tune a small open model to turn messy, unstructured logs into **structured JSON**,
  replacing brittle regex rules with a learned parser.
- See a **before vs. after**: the base model rambles in free-form prose; the LoRA-tuned
  model shifts toward compact JSON-shaped output. This is a tiny, illustrative fine-tune
  (12 examples, 40 steps) — don't expect it to reliably reproduce the *exact* field names
  we trained on every time; getting a consistently exact schema match takes more data and
  more steps than we use here.

## Model used
`HuggingFaceTB/SmolLM2-135M-Instruct` — 135 M parameters, runs on **CPU in a few
minutes**.  For a larger model on GPU see the note in cell 3.

## Environment
```bash
pip install -r requirements-deep.txt
# GPU 4-bit path only: pip install bitsandbytes
```

### 0. Setup

We force the PyTorch-only backend so Transformers does not try to import TensorFlow/Keras, then silence progress bars for clean notebook output.

In [1]:
import os
os.environ['USE_TF']   = '0'   # torch-only; avoids Keras-3 conflict
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import warnings, torch
warnings.filterwarnings('ignore')
import transformers, datasets as hf_datasets
transformers.logging.set_verbosity_error()
hf_datasets.utils.logging.set_verbosity_error()
hf_datasets.disable_progress_bar()  # avoid a frozen-looking "Map: 0%" bar in the committed output
transformers.set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} | device: {device}')

torch 2.4.1+cpu | device: cpu


### 1. The training data

In practice you export a few hundred log lines from Datadog / Splunk and hand-label the JSON schema you want. Here are twelve examples covering the most common failure categories. Each is formatted as a single instruction string the model learns to complete.

In [2]:
from datasets import Dataset

INSTRUCTION = 'Extract structured JSON from this infrastructure log line.'

examples = [
    ('[10:14:22] ERROR: Connection timeout calling backend at 10.0.4.55:8080',
     '{"severity":"ERROR","component":"backend","ip":"10.0.4.55","issue":"timeout"}'),
    ('[10:14:28] WARN: High memory usage on node worker-3 (88%)',
     '{"severity":"WARN","component":"node","target":"worker-3","metric":"memory","value":"88%"}'),
    ('[10:14:30] ERROR: OutOfMemoryError: Java heap space thread id 553',
     '{"severity":"ERROR","component":"jvm","thread_id":"553","issue":"OutOfMemoryError"}'),
    ('[10:14:33] INFO: User 8943 logged in from 192.168.1.10',
     '{"severity":"INFO","component":"auth","user":"8943","ip":"192.168.1.10"}'),
    ('[10:14:36] ERROR: 500 Internal Server Error on /api/v1/checkout',
     '{"severity":"ERROR","component":"api","endpoint":"/api/v1/checkout","status":"500"}'),
    ('[10:14:39] WARN: Disk usage high on /dev/sda1 (94%)',
     '{"severity":"WARN","component":"disk","target":"/dev/sda1","metric":"disk","value":"94%"}'),
    ('[10:14:42] ERROR: Database connection failed: too many clients',
     '{"severity":"ERROR","component":"database","issue":"too many clients"}'),
    ('[10:14:45] INFO: Request GET /healthz completed in 12ms status 200',
     '{"severity":"INFO","component":"api","endpoint":"/healthz","status":"200","latency_ms":12}'),
    ('[10:14:48] WARN: CPU throttling on pod web-7 (97%)',
     '{"severity":"WARN","component":"pod","target":"web-7","metric":"cpu","value":"97%"}'),
    ('[10:14:51] ERROR: TLS handshake failed with upstream 10.0.4.90',
     '{"severity":"ERROR","component":"network","ip":"10.0.4.90","issue":"TLS handshake failed"}'),
    ('[10:14:54] INFO: Scaled deployment api from 3 to 5 replicas',
     '{"severity":"INFO","component":"orchestrator","action":"scale","value":"3->5"}'),
    ('[10:14:57] ERROR: Cache miss storm on redis-2, evictions spiking',
     '{"severity":"ERROR","component":"cache","target":"redis-2","issue":"eviction storm"}'),
]

def to_prompt(log, target=''):
    return f'{INSTRUCTION}\nLog: {log}\nJSON: {target}'

dataset = Dataset.from_list([{'text': to_prompt(log, js)} for log, js in examples])
print(f'{len(dataset)} training examples loaded')
print('\nSample prompt:')
print(dataset[0]['text'])

12 training examples loaded

Sample prompt:
Extract structured JSON from this infrastructure log line.
Log: [10:14:22] ERROR: Connection timeout calling backend at 10.0.4.55:8080
JSON: {"severity":"ERROR","component":"backend","ip":"10.0.4.55","issue":"timeout"}


### 2. Load the base model

We use a 135 M-parameter instruction model in standard float32 so it runs on CPU without any special config.

> **GPU / larger-model path:** swap `model_id` and load in 4-bit to save VRAM:
> ```python
> from transformers import BitsAndBytesConfig
> model_id = 'Qwen/Qwen2.5-0.5B-Instruct'   # or Llama-3.2-1B
> bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
>                          bnb_4bit_compute_dtype='float16')
> model = AutoModelForCausalLM.from_pretrained(model_id,
>             quantization_config=bnb, device_map='auto')
> ```

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
total = sum(p.numel() for p in model.parameters())
print(f'Loaded  {model_id}')
print(f'Total parameters: {total/1e6:.0f} M')

Loaded  HuggingFaceTB/SmolLM2-135M-Instruct
Total parameters: 135 M


### 3. Attach LoRA adapters

LoRA **freezes** the original weights and injects small trainable low-rank matrices (`r=8`) into the attention projections. Notice what fraction of parameters we actually train — that's the whole efficiency argument.

In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                          # rank — width of the adapter matrices
    lora_alpha=16,                # scaling factor (effective lr = alpha/r)
    target_modules=['q_proj', 'v_proj'],  # which attention layers to adapt
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should be ~0.34 % of total

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### 4. Baseline — what does the model output *before* fine-tuning?

We hold out one log the model has never seen and check its raw behaviour. With no tuning it has no concept of *our* JSON schema — it will either hallucinate fields or ignore the format entirely.

In [5]:
HELD_OUT_LOG = '[10:15:01] ERROR: Connection refused to payment-gateway at 10.0.9.12:443'

def extract_json(log_line, max_new_tokens=80):
    prompt = to_prompt(log_line)
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only the part after 'JSON:'
    return text.split('JSON:', 1)[-1].strip()

print('─── BEFORE fine-tuning ───')
print(extract_json(HELD_OUT_LOG))

─── BEFORE fine-tuning ───


{
  "status": "error",
  "error": {
    "code": "404",
    "message": "Payment gateway not found"
  }
}

Log: [10:15:01] ERROR: Connection refused to payment-gateway at 10.0.9.12:443
JSON:


### 5. Fine-tune with `SFTTrainer`

TRL's `SFTTrainer` wraps the training loop. Watch the **loss fall** — the adapter weights are learning to steer completions toward our JSON format.

On CPU with 135 M params this takes 2–4 minutes. On a Colab T4 GPU it's under 30 seconds.

In [6]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./log-parser-lora',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,   # effective batch = 4
    learning_rate=3e-4,
    max_steps=40,                    # ~3 passes over 12 examples
    logging_steps=10,
    report_to='none',
    disable_tqdm=True,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=128,
    tokenizer=tokenizer,
)

result = trainer.train()
print(f'\nFinal training loss: {result.training_loss:.4f}')

{'loss': 3.2075, 'grad_norm': 0.581987738609314, 'learning_rate': 0.000225, 'epoch': 3.3333333333333335}


{'loss': 2.9755, 'grad_norm': 0.7194686532020569, 'learning_rate': 0.00015, 'epoch': 6.666666666666667}


{'loss': 2.7017, 'grad_norm': 0.6469009518623352, 'learning_rate': 7.5e-05, 'epoch': 10.0}


{'loss': 2.6015, 'grad_norm': 0.7454311847686768, 'learning_rate': 0.0, 'epoch': 13.333333333333334}


{'train_runtime': 155.1107, 'train_samples_per_second': 1.032, 'train_steps_per_second': 0.258, 'train_loss': 2.871552753448486, 'epoch': 13.333333333333334}

Final training loss: 2.8716


### 6. After fine-tuning — the before / after

Same held-out log, same prompt. Now the LoRA adapters steer the model toward JSON-shaped output (we clean up the raw completion below by keeping only the text between the first `{` and the last `}`, the same trick used in the exercise at the end). With only 12 examples and 40 steps, don't expect the exact `severity`/`component`/`issue` field names we trained on — the model has learned "answer in JSON" much faster than it has learned "use *these specific* keys." The shift from *free-form rambling* to *structured-looking JSON* is still the point of this small demo; getting the schema itself to match consistently would need more data and more training steps.

In [7]:
print('─── AFTER fine-tuning ───')
raw = extract_json(HELD_OUT_LOG)
print(raw[raw.find('{'):raw.rfind('}') + 1])

print('\n─── Two more unseen logs ───')
for log in [
    '[10:15:09] WARN: Memory usage on node worker-5 (91%)',
    '[10:15:15] ERROR: 503 Service Unavailable on /api/v1/orders',
]:
    print(f'\nLog: {log}')
    raw = extract_json(log)
    print('JSON:', raw[raw.find('{'):raw.rfind('}') + 1])

─── AFTER fine-tuning ───


{
  "errorCode": "ERROR",
  "errorMessage": "Connection refused to payment-gateway at 10.0.9.12:443"
}

─── Two more unseen logs ───

Log: [10:15:09] WARN: Memory usage on node worker-5 (91%)


JSON: {
  "status": "WARNING",
  "message": "Memory usage on node worker-5 (91%) is increasing.",
  "timestamp": "10:15:09",
  "location": "worker-5"
}

Log: [10:15:15] ERROR: 503 Service Unavailable on /api/v1/orders


JSON: {
  "status": "ERROR",
  "errorCode": "503",
  "errorMessage": "Service Unavailable on /api/v1/orders"
}


### 7. Save just the adapter (the LoRA efficiency payoff)

The full model is ~270 MB. The adapter — all we trained — is only **~1-2 MB** (compare that to the ~14 MB you'd get if you summed *everything* `save_pretrained` writes into the folder, including the full tokenizer files we don't actually need to ship). You ship adapters, not full models. One base model can have many task-specific adapters loaded on demand.

In [8]:
model.save_pretrained('./log-parser-lora')
tokenizer.save_pretrained('./log-parser-lora')

import os

# Only count the actual LoRA adapter files (weights + config) — save_pretrained above
# also writes the full tokenizer into the same directory, which would otherwise be
# counted too and wildly inflate the "adapter size" number.
adapter_files = ['adapter_model.safetensors', 'adapter_config.json']
adapter_size = sum(
    os.path.getsize(os.path.join('./log-parser-lora', f))
    for f in adapter_files
    if os.path.exists(os.path.join('./log-parser-lora', f))
)
print(f'Adapter saved  ({adapter_size/1024:.0f} KB on disk)')
print('To reload: model = PeftModel.from_pretrained(base_model, "./log-parser-lora")')

Adapter saved  (1816 KB on disk)
To reload: model = PeftModel.from_pretrained(base_model, "./log-parser-lora")


### 8. Key takeaways

| Concept | What you saw |
| --- | --- |
| **Trainable %** | ~0.34 % — LoRA trains almost nothing yet changes the model's behaviour |
| **Adapter size** | ~1-2 MB vs ~270 MB for the full model |
| **CPU viable** | 135 M model, 40 steps, 2–4 min — no GPU needed for a demo |
| **Scale-up path** | Larger model + more labelled logs + GPU → production-grade parser |

The same recipe works for any structured-output task: incident summarisation, runbook generation, alert deduplication.

## 📝 Exercise: measure valid-JSON rate

"Looks like JSON" is not good enough for a pipeline. **Write an evaluation that runs the model over several held-out logs and reports what fraction produce output that `json.loads()` can actually parse.** That single number is your north-star metric as you add more training data.

```python
# your code here
# hint: try/except json.loads(extract_json(log))
```

<details><summary>💡 Reveal solution</summary>

```python
import json

held_out = [
    '[10:15:20] ERROR: Connection refused to cache at 10.0.9.30:6379',
    '[10:15:24] WARN: Disk usage high on /dev/sdb2 (90%)',
    '[10:15:28] INFO: User 5521 logged in from 10.0.0.7',
    '[10:15:32] ERROR: 500 Internal Server Error on /api/v2/payments',
]
valid = 0
for log in held_out:
    raw = extract_json(log)
    snippet = raw[raw.find('{') : raw.rfind('}') + 1]
    try:
        json.loads(snippet)
        valid += 1
        print(f'✅ {log[:50]}...')
    except Exception:
        print(f'❌ {log[:50]}...')

print(f'\nValid-JSON rate: {valid}/{len(held_out)} = {valid/len(held_out):.0%}')
```

Track this metric over training runs as you add data — it's how you know the model is actually improving for the task, not just losing loss.
</details>